In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster
import requests
import logging
from pathlib import Path
import json

# Configure logging and style
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
plt.style.use('seaborn')

# Constants
PROXIMITY_THRESHOLD_METERS = 100
EARTH_RADIUS_METERS = 6371000
BARCELONA_PARKING_URL = "https://opendata-ajuntament.barcelona.cat/data/api/action/datastore_search"
PARKING_RESOURCE_ID = "1d6c814c-70ef-4147-aa16-a49ddb952f72"  # BSM parking facilities

# Barcelona geographical bounds
BARCELONA_BOUNDS = {
    'lat_min': 41.35,
    'lat_max': 41.45,
    'lon_min': 2.10,
    'lon_max': 2.23
}

def fetch_barcelona_parking_data():
    """Fetch parking facility data from Barcelona Open Data portal."""
    logger.info("Fetching Barcelona parking data...")
    
    try:
        params = {
            'resource_id': PARKING_RESOURCE_ID,
            'limit': 1000
        }
        
        response = requests.get(BARCELONA_PARKING_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if not data.get('success'):
            raise ValueError("API request was not successful")
            
        records = data['result']['records']
        logger.info(f"Retrieved {len(records)} records from API")
        
        # Create DataFrame
        df = pd.DataFrame(records)
        
        # Parse coordinates
        def parse_coordinates(coord_str):
            if pd.isna(coord_str):
                return pd.Series({'lon': None, 'lat': None})
            try:
                coords = coord_str.split(',')
                if len(coords) >= 2:
                    return pd.Series({
                        'lon': float(coords[0]),
                        'lat': float(coords[1])
                    })
            except (ValueError, IndexError) as e:
                logger.warning(f"Error parsing coordinates: {coord_str} - {e}")
            return pd.Series({'lon': None, 'lat': None})
            
        coord_df = df['Coordenades'].apply(parse_coordinates)
        df = pd.concat([df, coord_df], axis=1)
        
        # Add metadata
        df['parking_type'] = df['tipus_estacionament']
        df['capacity'] = pd.to_numeric(df['places'] if 'places' in df.columns else pd.Series([0] * len(df)), errors='coerce').fillna(0)
        df['name'] = df['nom']
        
        # Drop rows with missing coordinates
        df = df.dropna(subset=['lat', 'lon'])
        
        return df
        
    except Exception as e:
        logger.error(f"Error fetching Barcelona parking data: {e}")
        raise

def merge_datasets(bsm_data: pd.DataFrame, siu_data: pd.DataFrame, proximity_threshold: float) -> tuple[pd.DataFrame, dict]:
    """Match SIU points with nearest parking facilities."""
    
    # Convert coordinates to radians
    bsm_coords = np.radians(bsm_data[['lat', 'lon']].values)
    siu_coords = np.radians(siu_data[['lat', 'lon']].values)

    # Build BallTree
    tree = BallTree(bsm_coords, metric='haversine')

    # Find nearest neighbors
    distances, indices = tree.query(siu_coords, k=3)

    # Convert distances to meters
    distances_meters = distances * EARTH_RADIUS_METERS

    # Filter matches within threshold
    matched_indices = distances_meters[:, 0] < proximity_threshold
    matched_bsm_indices = indices[matched_indices, 0]
    
    if not matched_indices.any():
        logger.warning("No matches found!")
        return pd.DataFrame(), {}

    # Create matched dataframes
    matched_siu = siu_data[matched_indices].reset_index(drop=True)
    matched_bsm = bsm_data.iloc[matched_bsm_indices].reset_index(drop=True)
    
    # Add distance and quality information
    matched_siu['distance_to_parking'] = distances_meters[matched_indices, 0]
    matched_siu['match_quality'] = pd.cut(
        matched_siu['distance_to_parking'],
        bins=[0, 25, 50, 100, float('inf')],
        labels=['Excellent', 'Good', 'Fair', 'Poor']
    )
    
    # Rename BSM columns
    matched_bsm = matched_bsm.rename(columns={
        'lat': 'parking_lat',
        'lon': 'parking_lon',
        'name': 'facility_name'
    })
    
    # Merge datasets
    merged_data = pd.concat([matched_siu, matched_bsm], axis=1)
    
    # Calculate statistics
    stats = {
        'total_matches': len(merged_data),
        'unique_facilities': len(merged_data['facility_name'].unique()),
        'match_quality': merged_data['match_quality'].value_counts().to_dict(),
        'distance_stats': {
            'min': merged_data['distance_to_parking'].min(),
            'max': merged_data['distance_to_parking'].max(),
            'mean': merged_data['distance_to_parking'].mean(),
            'median': merged_data['distance_to_parking'].median()
        }
    }
    
    return merged_data, stats

def visualize_results(merged_data: pd.DataFrame, bsm_data: pd.DataFrame):
    """Create visualizations of the matching results."""
    
    # 1. Distance Distribution
    plt.figure(figsize=(12, 6))
    sns.histplot(data=merged_data, x='distance_to_parking', bins=50)
    plt.title('Distribution of Distances to Nearest Parking')
    plt.xlabel('Distance (meters)')
    plt.ylabel('Count')
    plt.show()

    # 2. Match Quality Distribution
    plt.figure(figsize=(10, 6))
    sns.countplot(data=merged_data, x='match_quality', order=['Excellent', 'Good', 'Fair', 'Poor'])
    plt.title('Distribution of Match Quality')
    plt.xticks(rotation=45)
    plt.show()

    # 3. Facility Utilization
    facility_usage = merged_data.groupby('facility_name').size().sort_values(ascending=False)
    plt.figure(figsize=(15, 6))
    facility_usage.head(20).plot(kind='bar')
    plt.title('Top 20 Most Matched Parking Facilities')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # 4. Interactive Map
    m = folium.Map(
        location=[merged_data['lat'].mean(), merged_data['lon'].mean()],
        zoom_start=13
    )
    
    # Add parking facilities
    facility_group = MarkerCluster(name='Parking Facilities')
    for idx, row in bsm_data.iterrows():
        folium.Marker(
            [row['lat'], row['lon']],
            popup=f"Facility: {row['name']}<br>Capacity: {row['capacity']}",
            icon=folium.Icon(color='red')
        ).add_to(facility_group)
    facility_group.add_to(m)
    
    # Add heatmap of matches
    HeatMap(merged_data[['lat', 'lon']].values.tolist()).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    return m

# Load and process data
logger.info("Loading data files...")
bsm_data = fetch_barcelona_parking_data()
siu_data = pd.read_csv('path/to/your/siu_data.csv')

# Display basic information
print("\nParking Facilities Data:")
print(bsm_data.info())
print("\nSIU Data:")
print(siu_data.info())

# Perform matching
merged_data, stats = merge_datasets(bsm_data, siu_data, PROXIMITY_THRESHOLD_METERS)

# Display statistics
print("\nMatching Statistics:")
print(json.dumps(stats, indent=2))

# Create visualizations
m = visualize_results(merged_data, bsm_data)
display(m)  # This will show the interactive map in the notebook

OSError: 'seaborn' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)